# Notebook 1 — Building the Surface Code by Hand

*Part of **QEC Explorer**. This is the foundation notebook: everything else is built on top of it, so the goal here is simple — **earn your trust in the math**, one printed line at a time.*

By the end you will have, in ~150 lines of plain Python, built a real distance-$d$ rotated surface code: its data qubits, its stabilizers, and the syndrome-reading and logical-error-detection logic. No quantum-computing libraries — just NumPy and `for` loops you can step through.

> ### 📺 Open the visualizer side-by-side
> This notebook is the math *behind* the **Module 1 — Detection** tool. Keep it open in another tab and compare as you go:
> **→ [QEC Explorer · Module 1 (live)](https://github.com/kondshk/QEC-Explorer)**
>
> Everything you build here uses the **identical coordinate and checkerboard convention** as that tool. We'll prove it with assertion cells — when they print ✅, your hand-built code agrees exactly with the live version.

**What this notebook deliberately leaves out** (so the bare math stays the point):
- **Decoding** — *which* correction to apply — is Notebook 2 (and the Module 2 decoder-race tool).
- **Noise models** (Notebook 3) and **Qiskit/Stim circuits** come later. Here, error correction is pure classical bookkeeping over qubit grids.

---
## Setup

Colab already has these. Run this once.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

np.random.seed(0)
plt.rcParams["figure.dpi"] = 110
print("NumPy", np.__version__, "— ready.")

---
## 1 · Why bother correcting errors at all?

Physical qubits are **noisy**: left alone, a qubit's state decays and random bit-flips and phase-flips creep in within microseconds. To compute anything useful we need the information to survive far longer than any single physical qubit does.

The obvious classical trick — **copy the bit three times and take a majority vote** — is *forbidden* for quantum information by the **no-cloning theorem**: there is no operation that duplicates an unknown quantum state. So we can't make backup copies.

The way out, discovered in the 1990s, is to **spread one logical qubit's information across many physical qubits** in an entangled pattern, so that *no single qubit holds the information* and *local errors can be detected and undone without ever measuring (and thus destroying) the logical state itself.* The **surface code** is the most practical such scheme, and it's what we'll build.

The promise is the **threshold theorem**: if your physical error rate $p$ is below some critical value $p_\text{th}$, then making the code **bigger** (larger *distance* $d$) makes the logical error rate drop *exponentially*. Below threshold, bigger = safer. The plot below is just a sketch of that behavior to motivate the rest — the real derivation waits for Notebook 4.

In [ ]:
# A simple sub-threshold scaling sketch:  p_logical ~ A * (p / p_th)^((d+1)/2)
# This is the standard heuristic form (a code of distance d corrects (d-1)/2
# errors, so the leading failure term scales like p^((d+1)/2)). It is ONLY a
# motivational cartoon here — Notebook 4 derives the real curve.
p_th = 0.10                     # a representative surface-code threshold (~10%)
p_phys = np.logspace(-3, -0.7, 200)   # physical error rate, ~0.001 .. ~0.2

plt.figure(figsize=(6, 4.2))
for d in (3, 5, 7):
    A = 0.5
    p_log = A * (p_phys / p_th) ** ((d + 1) / 2)
    plt.loglog(p_phys, p_log, label=f"d = {d}")

plt.axvline(p_th, ls="--", color="0.5", lw=1)
plt.text(p_th * 1.05, 1e-6, "threshold $p_{th}$", color="0.4", fontsize=9)
plt.plot(p_phys, p_phys, "k:", lw=1, label="break-even (no code)")
plt.xlabel("physical error rate  $p$")
plt.ylabel("logical error rate  $p_L$")
plt.title("Why distance helps — but only below threshold")
plt.legend(frameon=False, fontsize=9)
plt.ylim(1e-9, 1)
plt.tight_layout()
plt.show()

Read the plot left-to-right. **To the left of the dashed threshold line**, the higher-distance curves plunge *below* the dotted break-even line and below each other: more qubits → far fewer logical errors. **To the right of threshold**, the order flips — bigger codes are *worse*, because you've added more faulty parts than the code can fix. That crossover is the whole game. Now let's build the thing that makes it possible.

---
## 2 · The data qubits

The rotated surface code lays its **data qubits** on a simple $d \times d$ grid. We'll use the **simplest possible representation**: a list of `(row, col)` tuples. (In production you'd reach for a dict of objects; resist that here — the point is to *see* the structure.)

**Coordinate convention — read this carefully, everything downstream depends on it:**
- `r` is the **row**, increasing **downward** (row 0 at the top).
- `c` is the **column**, increasing **to the right** (column 0 at the left).
- This is exactly the convention the **Module 1 visualizer** uses, so a qubit at `(1, 2)` here is the same qubit you'd click there.

In [ ]:
def build_data_qubits(d):
    # Return the d*d data qubits as a list of (row, col) tuples.
    # Matches lattice-core.js: for r in 0..d-1, for c in 0..d-1.
    data = []
    for r in range(d):
        for c in range(d):
            data.append((r, c))
    return data

d = 3
data = build_data_qubits(d)
print(f"d = {d}: {len(data)} data qubits")
print(data)

In [ ]:
def plot_qubits(d, data, title="Data qubits"):
    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    for (r, c) in data:
        ax.plot(c, r, "o", ms=16, color="#3a3a5a", mec="#8888aa", mew=1.5, zorder=3)
        ax.text(c, r, f"{r},{c}", color="white", fontsize=7.5,
                ha="center", va="center", zorder=4)
    ax.set_aspect("equal")
    ax.invert_yaxis()                  # row 0 on top  ->  matches the visualizer
    ax.set_xlabel("column  c  →")
    ax.set_ylabel("←  row  r")
    ax.set_title(f"{title}  (d = {d})")
    ax.set_xticks(range(d)); ax.set_yticks(range(d))
    ax.grid(True, ls=":", color="0.85")
    plt.tight_layout(); plt.show()

plot_qubits(d, data)

Note `ax.invert_yaxis()`: we flip the vertical axis so **row 0 sits at the top**, matching how the visualizer draws it. If you ever see your notebook and the web tool disagree about *where* a qubit is, this line is the usual culprit.

---
## 3 · Stabilizers — the checkerboard rule

This is the conceptual heart of the whole code, so we go slowly.

A **stabilizer** is a measurement that asks one **yes/no question** about a small group of neighboring data qubits — *"is there an odd number of errors among you?"* — **without revealing the qubits' actual quantum state.** That last part is the magic: a stabilizer measurement returns a single classical bit (the answer to the parity question) and collapses *nothing* about the logical information. We get to keep asking these questions over and over to watch for errors.

Geometrically, each stabilizer lives on a **plaquette** — a little square sitting *between* data qubits, centered at a half-integer position like $(1.5, 0.5)$. A bulk plaquette touches the **4** data qubits at its corners; a plaquette on the boundary touches only **2**.

There are two **types**, laid out like a checkerboard:
- **Z-type** stabilizers detect **X errors** (bit-flips).
- **X-type** stabilizers detect **Z errors** (phase-flips).

The rotated code's defining shape comes from one boundary rule: **Z-type half-plaquettes survive only on the left/right edges; X-type half-plaquettes survive only on the top/bottom edges.** Everything else is bulk weight-4. Here is that logic, written out plainly — this is a direct port of `buildCode()` in `lattice-core.js`, just unrolled so you can read every branch:

In [ ]:
def build_stabilizers(d):
    # Build the rotated-surface-code stabilizers for distance d.
    #
    # Direct, un-optimized port of lattice-core.js buildCode():
    #   - candidate plaquette centers sit at (R+0.5, C+0.5) for R,C in -1..d-1
    #   - type by checkerboard parity: (R+C) even -> 'Z', odd -> 'X'
    #   - a plaquette touches the data qubits at its 4 integer corners that
    #     actually exist on the grid
    #   - boundary rule: keep a weight-2 plaquette only if it's the right type
    #     for that edge (Z on left/right, X on top/bottom); drop weight 1 & 3.
    # Each stabilizer is a dict: {type, center=(cx,cy), data=[(r,c), ...]}.
    stabs = []
    for R in range(-1, d):
        for C in range(-1, d):
            stype = "Z" if (R + C) % 2 == 0 else "X"   # Z detects X, X detects Z

            # the (up to) four corner data qubits that exist on the grid
            corners = [(R, C), (R, C + 1), (R + 1, C), (R + 1, C + 1)]
            cand = [(rr, cc) for (rr, cc) in corners
                    if 0 <= rr < d and 0 <= cc < d]
            if len(cand) == 0:
                continue

            on_top_bottom = (R == -1 or R == d - 1)
            on_left_right = (C == -1 or C == d - 1)

            if len(cand) == 2:
                # weight-2 boundary plaquette: keep only the matching type/edge
                if stype == "Z" and not on_left_right:
                    continue
                if stype == "X" and not on_top_bottom:
                    continue
            if len(cand) not in (2, 4):
                continue   # weight 1 or 3 candidate sets are not valid stabilizers

            stabs.append({
                "type": stype,
                "center": (C + 0.5, R + 0.5),   # (cx, cy) = (col, row), matches viz
                "data": cand,
            })
    return stabs

stabs = build_stabilizers(d)
print(f"d = {d}: {len(stabs)} stabilizers "
      f"({sum(s['type']=='X' for s in stabs)} X-type, "
      f"{sum(s['type']=='Z' for s in stabs)} Z-type)\n")
for i, s in enumerate(stabs):
    print(f"[{i}] {s['type']}  center={s['center']}  data={s['data']}")

### Plot it — and compare to the live tool

Below we draw the lattice with stabilizers overlaid: **teal squares are Z-type** (X-error detectors), **pink squares are X-type** (Z-error detectors).

> 🔍 **Compare this directly to [Module 1 (live)](https://github.com/kondshk/QEC-Explorer).** Same qubit positions, same two-color checkerboard, same boundary half-plaquettes. If you injected no errors in the live tool, its lattice should look like this picture.

In [ ]:
def plot_lattice(d, data, stabs, title="Surface code lattice"):
    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    Z_FILL, X_FILL = "#2ec4a0", "#e45d9a"   # teal / pink, matching the visualizer

    # draw plaquettes first (under the qubits)
    for s in stabs:
        pts = [(c, r) for (r, c) in s["data"]]        # (x=col, y=row)
        cx, cy = s["center"]
        # order corners around the center so the polygon doesn't self-cross
        pts.sort(key=lambda p: np.arctan2(p[1] - cy, p[0] - cx))
        fill = Z_FILL if s["type"] == "Z" else X_FILL
        poly = plt.Polygon(pts, closed=True, facecolor=fill, alpha=0.18,
                           edgecolor=fill, lw=1.4, zorder=1)
        ax.add_patch(poly)

    # bonds between adjacent data qubits
    for (r, c) in data:
        if (r, c + 1) in data: ax.plot([c, c + 1], [r, r], color="0.8", lw=1, zorder=1)
        if (r + 1, c) in data: ax.plot([c, c], [r, r + 1], color="0.8", lw=1, zorder=1)

    # data qubits on top
    for (r, c) in data:
        ax.plot(c, r, "o", ms=15, color="#2a2a3a", mec="#43435f", mew=1.4, zorder=3)

    # tiny legend
    ax.add_patch(plt.Polygon([(-0.9, d-0.4), (-0.6, d-0.4), (-0.6, d-0.1), (-0.9, d-0.1)],
                             facecolor=Z_FILL, alpha=0.3, edgecolor=Z_FILL))
    ax.text(-0.5, d-0.25, "Z-type (detects X)", fontsize=8, va="center")
    ax.add_patch(plt.Polygon([(-0.9, d+0.1), (-0.6, d+0.1), (-0.6, d+0.4), (-0.9, d+0.4)],
                             facecolor=X_FILL, alpha=0.3, edgecolor=X_FILL))
    ax.text(-0.5, d+0.25, "X-type (detects Z)", fontsize=8, va="center")

    ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_xlabel("column  c  →"); ax.set_ylabel("←  row  r")
    ax.set_title(f"{title}  (d = {d})")
    ax.set_xlim(-1.2, d - 0.3); ax.set_ylim(d + 0.6, -1.0)
    plt.tight_layout(); plt.show()

plot_lattice(d, data, stabs)

### ✏️ Exercise 3.1 — the $d^2 - 1$ rule

Change `d` to **5** in the cell below and run it. **How many stabilizers do you get?**

Then think about *why*. A distance-$d$ rotated code encodes **1** logical qubit into $d^2$ physical qubits, and it takes $d^2 - 1$ independent stabilizers to pin down that single logical degree of freedom. (Count: physical qubits − logical qubits = stabilizers.) Check your number against the formula.

In [ ]:
# --- Exercise 3.1: try d = 5, then d = 7. Predict the count before you run! ---
for d_try in (3, 5, 7):
    s = build_stabilizers(d_try)
    print(f"d = {d_try}:  {len(s):>2} stabilizers   "
          f"(d^2 - 1 = {d_try**2 - 1})   match: {len(s) == d_try**2 - 1}")

---
## 4 · Syndromes — reading the stabilizers

An **error** on a data qubit is a Pauli: an **X** (bit-flip), a **Z** (phase-flip), or a **Y** (both). We represent an error set as a dict mapping `(r, c)` → `{"x": bool, "z": bool}`.

A stabilizer **fires** (its syndrome bit = 1) when an **odd number** of the errors it touches anticommute with it:
- a **Z-type** stabilizer fires on an **odd count of X** components among its qubits,
- an **X-type** stabilizer fires on an **odd count of Z** components.

The collection of all stabilizer bits is the **syndrome** — the *only* information a decoder ever gets to see. Here is the parity computation, a direct port of `computeSyndrome()`:

In [ ]:
def compute_syndrome(stabs, errors):
    # Return a list of 0/1, one per stabilizer, given an error dict.
    # errors: { (r,c): {'x':bool,'z':bool}, ... }.  Port of computeSyndrome().
    syndrome = []
    for s in stabs:
        parity = 0
        for (r, c) in s["data"]:
            e = errors.get((r, c))
            if not e:
                continue
            if s["type"] == "Z" and e.get("x"):   # Z-stab sees X errors
                parity ^= 1
            if s["type"] == "X" and e.get("z"):   # X-stab sees Z errors
                parity ^= 1
        syndrome.append(parity)
    return syndrome

# Inject a single X error by hand and read the stabilizers.
errors = {(1, 1): {"x": True, "z": False}}
syn = compute_syndrome(stabs, errors)

print("Error: single X at (1,1)")
print("Syndrome:", "".join(map(str, syn)))
print("\nStabilizers that fired:")
for i, bit in enumerate(syn):
    if bit:
        print(f"  [{i}] {stabs[i]['type']}-type, center {stabs[i]['center']}, "
              f"touches {stabs[i]['data']}")

A single X error lights up the **Z-type** stabilizers on either side of it — two fired detectors straddling the qubit. That pair of lit detectors is the "signature" a decoder will later try to explain.

### ✏️ Exercise 4.1 — make an error *hide*

Syndromes don't always tell you exactly what happened. Find **two** X errors that, together, leave one particular Z-stabilizer **silent** even though both qubits are wrong.

Hint: a Z-stabilizer counts X errors *modulo 2*. What happens if **two** of the qubits it touches are both flipped?

In [ ]:
# --- Exercise 4.1: two errors that cancel a stabilizer's signal ---
# Stabilizer [1] is a Z-type plaquette touching (0,0),(0,1),(1,0),(1,1).
# Flip TWO of its four qubits and watch its bit stay 0 (even parity).
two_errors = {
    (0, 0): {"x": True, "z": False},
    (0, 1): {"x": True, "z": False},
}
syn2 = compute_syndrome(stabs, two_errors)
print("Two X errors at (0,0) and (0,1).")
print("Syndrome:", "".join(map(str, syn2)))
print(f"\nStabilizer [1] (the Z-plaquette over those qubits) fired? "
      f"{'YES' if syn2[1] else 'NO — its two errors cancelled in parity'}")
print("\nNotice: those two qubits are definitely in error, yet stabilizer [1]")
print("reports nothing. Syndromes reveal *parity*, not *location*. Hold onto")
print("that thought — it's exactly why decoding (Notebook 2) is hard.")

---
## 5 · Logical operators and the column-0 / row-0 trick

Here is the subtlest and most important idea in the notebook. Take a deep breath.

**Some error patterns are completely invisible to every stabilizer — yet they corrupt the stored logical qubit.** These are the **logical operators**, and they are the failures error correction actually exists to prevent.

The cleanest example: an **X error on an entire vertical column** of data qubits.

### Why a full column is silent

Walk a single Z-type stabilizer in the bulk: it touches a $2\times 2$ block of qubits, so it touches **two** qubits of any given column passing through it. A full column of X errors therefore flips an **even** number (two) of that stabilizer's qubits → **parity 0 → silent.** Every stabilizer it overlaps sees an *even* count and stays dark. The errors **cancel pairwise** across every plaquette. Let's watch that happen:

In [ ]:
# A full column-0 of X errors: inject X on every qubit in column c = 0.
col_errors = {(r, 0): {"x": True, "z": False} for r in range(d)}
syn_col = compute_syndrome(stabs, col_errors)

print("Full column-0 X errors:", sorted(col_errors.keys()))
print("Syndrome:", "".join(map(str, syn_col)),
      "  <- all zeros: every stabilizer is silent!\n")

# Show the pairwise cancellation: for each stabilizer, how many of its qubits
# are in the errored column?
print("Per-stabilizer count of errored qubits (must be EVEN to stay silent):")
for i, s in enumerate(stabs):
    touched = sum(1 for (r, c) in s["data"] if (r, c) in col_errors)
    if touched:
        print(f"  [{i}] {s['type']}  touches {touched} errored qubit(s)  "
              f"-> parity {touched % 2}")

Every overlap is **2** (even) → every bit stays 0. The syndrome is **identical to "no error at all."** And yet the logical qubit has been flipped. No amount of stabilizer information can distinguish this from a clean code — *that* is what makes it dangerous.

### Detecting it anyway: the column-0 / row-0 observables

We can still *detect* a logical flip if we're allowed to look at the actual error (which in a real device we can't — but here, building intuition, we can). The trick from `logicalStatus()`:

- A **logical X** flip = X-parity along **any single column** is odd. Column 0 is a valid representative, because for a syndrome-free X configuration this parity is a *homology invariant* (it doesn't matter which column you pick).
- A **logical Z** flip = Z-parity along **row 0**.

So: *if the syndrome is all-zero* **and** *(column-0 X-parity is odd OR row-0 Z-parity is odd)*, a logical error has slipped through.

In [ ]:
def logical_status(stabs, errors, d):
    # Port of logicalStatus(): is there an undetectable logical flip?
    # Only meaningful when the syndrome is clear; if any stabilizer fired,
    # the error is still 'detectable' and not (yet) a silent logical.
    syn = compute_syndrome(stabs, errors)
    if any(syn):
        return {"logical": False, "detectable": True}

    x_col0 = 0   # X-parity down column 0   -> flips logical Z
    z_row0 = 0   # Z-parity across row 0    -> flips logical X
    for r in range(d):
        e = errors.get((r, 0))
        if e and e.get("x"):
            x_col0 ^= 1
    for c in range(d):
        e = errors.get((0, c))
        if e and e.get("z"):
            z_row0 ^= 1
    return {"logical": (x_col0 == 1 or z_row0 == 1), "detectable": False}

print("full column-0 X :", logical_status(stabs, col_errors, d))
print("single  X(1,1)  :", logical_status(stabs, {(1,1):{'x':True,'z':False}}, d))
print("no error        :", logical_status(stabs, {}, d))

The full column reports `logical: True, detectable: False` — *silent but fatal.* The single error reports `detectable: True` — the stabilizers can see it, so it's correctable, not a logical failure. Exactly the distinction the **red "Logical error" banner** in Module 1 is showing you.

### ✏️ Exercise 5.1 — discover degeneracy for yourself

Now the punchline that connects straight to the **decoder-race tool (Module 2)**.

Inject the **single** error **Z at (1, 2)** and compute its syndrome. Then **hunt for a different single-qubit error that produces the *exact same* syndrome.** Try a few before peeking at the answer cell.

What you're about to find is called **degeneracy**: two genuinely different errors, same syndrome, no way to tell them apart from stabilizer data alone. It's not a bug — it's a fundamental feature of these codes, and it's the reason a decoder can be *right about the syndrome yet wrong about the world*.

In [ ]:
# --- Exercise 5.1, part 1: the syndrome of a single Z at (1,2) ---
target = {(1, 2): {"x": False, "z": True}}
syn_target = compute_syndrome(stabs, target)
print("Z(1,2) syndrome:", "".join(map(str, syn_target)))

# YOUR TURN: try editing `guess` to a DIFFERENT single-qubit error and see if
# its syndrome matches. (Scroll down for the answer once you've experimented.)
guess = {(0, 0): {"x": False, "z": True}}   # <-- change me and re-run
syn_guess = compute_syndrome(stabs, guess)
print("your guess     :", "".join(map(str, syn_guess)),
      " -> match!" if syn_guess == syn_target else " -> no match, keep hunting")

In [ ]:
# --- Exercise 5.1, part 2: brute-force ALL single-qubit errors (the reveal) ---
print(f"Searching every single-qubit error for the syndrome {''.join(map(str, syn_target))} ...\n")
matches = []
for (r, c) in data:
    for name, pauli in [("X", {"x": True, "z": False}),
                        ("Z", {"x": False, "z": True}),
                        ("Y", {"x": True, "z": True})]:
        syn = compute_syndrome(stabs, {(r, c): pauli})
        if syn == syn_target:
            matches.append((name, (r, c)))

for name, pos in matches:
    print(f"   {name} at {pos}")
print(f"\n{len(matches)} different single-qubit errors share this syndrome.")
print("Z(1,2) and Z(0,2) are indistinguishable from the syndrome alone — THIS is")
print("degeneracy. In Module 2, watch the matching decoder face exactly this Z(1,2)")
print("case: it must pick one, and the choice can quietly complete a logical operator.")

---
## 6 · Proof: this agrees with the live tool

Trust, but verify. The cell below re-derives the facts the **Module 1 / Module 2** tools depend on, straight from the functions you built above, and asserts each one. **All green ✅ means your hand-built surface code is byte-for-byte consistent with `lattice-core.js`** — the verified physics engine behind the visualizer. If you ever modify the build logic and break the convention, one of these will trip.

In [ ]:
def check(name, condition):
    print(f"  {'✅' if condition else '❌'}  {name}")
    assert condition, f"MISMATCH: {name}"

print("Re-deriving the live tool's invariants from your hand-built code:\n")

# --- counts & typing match the visualizer at every distance we ship ---
for dd in (3, 5, 7):
    sd = build_stabilizers(dd)
    check(f"d={dd}: stabilizer count is d^2-1 = {dd**2-1}", len(sd) == dd**2 - 1)
nx = sum(s["type"] == "X" for s in stabs)
nz = sum(s["type"] == "Z" for s in stabs)
check("d=3: 4 X-type and 4 Z-type stabilizers", nx == 4 and nz == 4)

# --- exact d=3 stabilizer layout, as emitted by lattice-core.js buildCode() ---
expected_d3 = [
    ("X", (0.5, -0.5), [(0,0),(0,1)]),
    ("Z", (0.5,  0.5), [(0,0),(0,1),(1,0),(1,1)]),
    ("X", (1.5,  0.5), [(0,1),(0,2),(1,1),(1,2)]),
    ("Z", (2.5,  0.5), [(0,2),(1,2)]),
    ("Z", (-0.5, 1.5), [(1,0),(2,0)]),
    ("X", (0.5,  1.5), [(1,0),(1,1),(2,0),(2,1)]),
    ("Z", (1.5,  1.5), [(1,1),(1,2),(2,1),(2,2)]),
    ("X", (1.5,  2.5), [(2,1),(2,2)]),
]
got_d3 = [(s["type"], s["center"], s["data"]) for s in build_stabilizers(3)]
check("d=3 stabilizer list matches lattice-core.js exactly", got_d3 == expected_d3)

# --- syndrome convention: single X fires Z-type detectors only ---
syn_x11 = compute_syndrome(stabs, {(1,1): {"x": True, "z": False}})
fired_types = {stabs[i]["type"] for i, b in enumerate(syn_x11) if b}
check("single X error fires only Z-type stabilizers", fired_types == {"Z"})

# --- the Z(1,2) degeneracy the decoder-race relies on ---
syn_a = compute_syndrome(stabs, {(1,2): {"x": False, "z": True}})
syn_b = compute_syndrome(stabs, {(0,2): {"x": False, "z": True}})
check("Z(1,2) and Z(0,2) share a syndrome (the Module-2 degeneracy)", syn_a == syn_b)
check("Z(1,2) and Z(0,2) are genuinely different errors", (1,2) != (0,2))

# --- logical-detection convention (column-0 X / row-0 Z) ---
col = {(r,0): {"x": True, "z": False} for r in range(3)}
check("full column-0 X is a silent logical error", logical_status(stabs, col, 3)["logical"] is True)
check("single X(1,1) is NOT a logical error", logical_status(stabs, {(1,1):{'x':True}}, 3)["logical"] is False)
check("empty error set is not a logical error", logical_status(stabs, {}, 3)["logical"] is False)

print("\n🎉  All invariants agree with the live QEC Explorer physics.")

---
## 7 · Wrap-up — and the bridge to Notebook 2

In about 150 lines of plain Python you built a real distance-$d$ rotated surface code: its data qubits, its checkerboard of stabilizers, the syndrome-parity readout, and the logical-operator check that separates a recoverable error from a fatal one. And you proved it matches the live visualizer exactly.

But notice what you **can't** do yet. You can *detect* that something is wrong — the syndrome lights up. You can even recognize a logical failure after the fact. What you **cannot** do is the thing that actually keeps a quantum computer alive: given only the lit detectors, **decide which correction to apply.** Exercise 4.1 showed syndromes hide *location*; Exercise 5.1 showed two different errors can give the *same* syndrome. So *choosing* the right fix is a genuine inference problem with no guaranteed-correct answer — that's **decoding**, and it's a much harder and more interesting problem than detection.

That's exactly what comes next:

- **→ Notebook 2 — Decoders** builds three real decoders (lookup table, minimum-weight matching, belief propagation) on top of *this* code, and shows where they agree, disagree, and fail.
- **→ [Module 1 · Detection (live)](https://github.com/kondshk/QEC-Explorer)** — the tool this notebook is the math behind. Inject the errors from Sections 4–5 and watch the syndromes you computed light up.
- **→ [Module 2 · Decoder Race (live)](https://github.com/kondshk/QEC-Explorer)** — watch three decoders race on the same syndrome. Feed it the **Z(1,2)** error from Exercise 5.1 and watch the degeneracy you discovered play out in real time.

You've built the foundation. Everything else stands on it.